In [0]:
import requests
import json
from datetime import datetime, timezone, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, ArrayType
import time
import uuid
import hashlib

In [0]:
# Catalog Configuration
CATALOG = "workspace"  # Update with your catalog

# Schema Names - following LMIP architecture
BRONZE_SCHEMA = "bronze"      # Raw ingestion layer (API snapshots, response logs)
METADATA_SCHEMA = "metadata"  # Pipeline run control
AUDIT_SCHEMA = "audit"        # Pipeline audit logs

# Bronze Tables (ingestion output)
BRONZE_JOB_SNAPSHOT = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_job_snapshot"
BRONZE_API_LOG = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_api_response_log"

# Metadata Tables
PIPELINE_RUN_CONTROL = f"{CATALOG}.{METADATA_SCHEMA}.pipeline_run_control"
SOURCE_CONFIG = f"{CATALOG}.{METADATA_SCHEMA}.source_config"

# Audit Tables
AUDIT_PIPELINE_RUNS = f"{CATALOG}.{AUDIT_SCHEMA}.audit_pipeline_runs"

In [0]:
def generate_payload_hash(payload_json):
    """Generate SHA-256 hash of payload for deduplication"""
    return hashlib.sha256(payload_json.encode('utf-8')).hexdigest()

def generate_snapshot_id(source_name, source_job_id, batch_id):
    """Generate unique snapshot identifier"""
    return f"{source_name}_{source_job_id}_{batch_id}"

def generate_batch_id():
    """Generate unique batch ID for ingestion run"""
    return str(uuid.uuid4())

def extract_arbeitnow_job_id(job):
    """Extract job ID from Arbeitnow job record"""
    return job.get('slug', job.get('id', str(uuid.uuid4())))

def extract_remotive_job_id(job):
    """Extract job ID from Remotive job record"""
    return str(job.get('id', uuid.uuid4()))

In [0]:
def prepare_bronze_snapshots(jobs_data, source_name, batch_id, extract_job_id_func):
    """Prepare individual job snapshots for Bronze layer"""
    if not jobs_data:
        return []
    
    bronze_records = []
    ingestion_timestamp = datetime.now(timezone.utc)
    ingestion_date = ingestion_timestamp.date()
    
    for job in jobs_data:
        source_job_id = extract_job_id_func(job)
        payload_json = json.dumps(job)
        payload_hash = generate_payload_hash(payload_json)
        snapshot_id = generate_snapshot_id(source_name, source_job_id, batch_id)
        
        bronze_records.append({
            'snapshot_id': snapshot_id,
            'source_name': source_name,
            'source_job_id': source_job_id,
            'batch_id': batch_id,
            'page_number': None,
            'page_size': None,
            'payload_json': payload_json,
            'payload_hash': payload_hash,
            'ingestion_timestamp': ingestion_timestamp,
            'ingestion_date': ingestion_date,
            'source_status_code': 200,
            'source_etag': None
        })
    
    return bronze_records

In [0]:
def ingest_to_bronze(source_name, fetch_function, api_url, extract_job_id_func,
                     log_api_response_func, start_pipeline_run_func, 
                     complete_pipeline_run_func, log_audit_func):
    """Main ingestion function - fetches API data and stores in Bronze layer
    
    Returns:
        Tuple of (success: bool, batch_id: str, records_count: int)
    """
    pipeline_name = f"bronze_ingestion_{source_name}"
    batch_id = generate_batch_id()
    start_time = time.time()
    
    # Start pipeline run in metadata
    run_control_sk = start_pipeline_run_func(pipeline_name, source_name, batch_id)
    print(f"\nPipeline: {pipeline_name}")
    print(f"Batch ID: {batch_id}")
    
    try:
        # Fetch data from API
        print(f"\n[1/3] Fetching data from {source_name}...")
        api_start = time.time()
        jobs_data, error = fetch_function()
        api_duration_ms = int((time.time() - api_start) * 1000)
        
        if error:
            # Log failed API response
            log_api_response_func(source_name, batch_id, api_url, 500, 
                           response_time_ms=api_duration_ms)
            
            duration = time.time() - start_time
            complete_pipeline_run_func(batch_id, 'FAILED')
            log_audit_func(batch_id, pipeline_name, 'FAILED', 
                          runtime_seconds=duration, error_message=error)
            print(f"  FAILED to fetch: {error}")
            return False, batch_id, 0
        
        records_fetched = len(jobs_data) if jobs_data else 0
        print(f"  Fetched {records_fetched} jobs in {api_duration_ms}ms")
        
        # Log successful API response
        log_api_response_func(source_name, batch_id, api_url, 200, 
                       response_time_ms=api_duration_ms)
        
        if not jobs_data:
            duration = time.time() - start_time
            complete_pipeline_run_func(batch_id, 'SUCCESS')
            log_audit_func(batch_id, pipeline_name, 'SUCCESS',
                          rows_read=0, rows_written=0, runtime_seconds=duration)
            print(f"  WARNING: No data returned")
            return True, batch_id, 0
        
        # Prepare Bronze job snapshots
        print(f"\n[2/3] Processing to Bronze snapshots...")
        bronze_records = prepare_bronze_snapshots(jobs_data, source_name, batch_id, extract_job_id_func)
        records_processed = len(bronze_records)
        print(f"  Processed {records_processed} snapshots")
        
        # Write to Bronze layer
        print(f"\n[3/3] Writing to {BRONZE_JOB_SNAPSHOT}...")
        from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, DateType
        bronze_schema = StructType([
            StructField("snapshot_id", StringType(), False),
            StructField("source_name", StringType(), False),
            StructField("source_job_id", StringType(), True),
            StructField("batch_id", StringType(), False),
            StructField("page_number", IntegerType(), True),
            StructField("page_size", IntegerType(), True),
            StructField("payload_json", StringType(), False),
            StructField("payload_hash", StringType(), False),
            StructField("ingestion_timestamp", TimestampType(), False),
            StructField("ingestion_date", DateType(), False),
            StructField("source_status_code", IntegerType(), True),
            StructField("source_etag", StringType(), True)
        ])
        
        df = spark.createDataFrame(bronze_records, schema=bronze_schema)
        df.write.mode('append').saveAsTable(BRONZE_JOB_SNAPSHOT)
        print(f"  Wrote {records_processed} records")
        
        duration = time.time() - start_time
        
        # Complete pipeline run
        complete_pipeline_run_func(batch_id, 'SUCCESS')
        log_audit_func(batch_id, pipeline_name, 'SUCCESS',
                      rows_read=records_fetched, 
                      rows_written=records_processed,
                      runtime_seconds=duration)
        
        print(f"\nSUCCESS - Ingested {records_processed} jobs to Bronze layer in {duration:.2f}s")
        return True, batch_id, records_processed
        
    except Exception as e:
        duration = time.time() - start_time
        error_msg = str(e)
        complete_pipeline_run_func(batch_id, 'FAILED')
        log_audit_func(batch_id, pipeline_name, 'FAILED',
                      runtime_seconds=duration, error_message=error_msg)
        print(f"\nFAILED - {error_msg}")
        return False, batch_id, 0